# Proposed method: evaluate the released checkpoint

Loads `checkpoints/best_p2_proposed_seed163.pth` and evaluates it on the test set (62,002 windows) without training.
The released predictions in `released_results/` were produced on GPU; CPU inference can differ by a few windows (floating-point order).

In [1]:
import os, sys, json
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve() / "common"))
import core as C
import numpy as np, pandas as pd, torch
print("torch", torch.__version__, "| device", C.DEVICE)

torch 2.7.1+cu118 | device cuda


In [2]:
@torch.no_grad()
def predict(net, X, bs=2048):
    net.eval()
    X = torch.from_numpy(X)
    return torch.cat([net(X[i:i + bs].to(C.DEVICE)).argmax(1).cpu() for i in range(0, len(X), bs)]).numpy()


def build(mode, S, S11):
    spec = C.ABL[mode]
    Sx = S if spec["data"] == "full" else S11
    nf = len(Sx["feats"])
    if spec["p1"] is None:
        net = C.PureBaselineMLP(nf, 4, spec["hidden"])
    else:
        net = C.DownstreamClassifier(C.make_encoder(spec["enc"], nf), 4, freeze_encoder=True)
    return net.to(C.DEVICE), Sx

In [3]:
S, _ = C.prepare("../data/features_21d", verbose=False)
net, _ = build("M5", S, None)
net.load_state_dict(torch.load("checkpoints/best_p2_proposed_seed163.pth", map_location="cpu"))
pred = predict(net, S["X_test"])
m = C.full_metrics(S["y_test"], pred)
print("correct %d / %d  acc %.4f%%  macro-F1 %.4f" % ((pred == S["y_test"]).sum(), len(pred), 100 * m["accuracy"], m["macro_f1"]))
print(m["report"])
ref = pd.read_csv("released_results/test_predictions_proposed_seed163.csv")
print("released (GPU) correct:", int((ref.true_label == ref.pred_label).sum()),
      "| windows that differ from the released predictions:", int((ref.pred_label.values != pred).sum()))

[OK]   self-test: one value changed by 1e-9 -> rejected (test: feature digest mismatch)
[OK]   self-test: columns reordered -> rejected (test: column names/order mismatch.)
correct 61658 / 62002  acc 99.4452%  macro-F1 0.9945
              precision    recall  f1-score   support

       Aging     0.9796    0.9986    0.9890     15501
       Fault     0.9986    0.9792    0.9888     15500
       Noise     1.0000    1.0000    1.0000     15501
      Normal     1.0000    1.0000    1.0000     15500

    accuracy                         0.9945     62002
   macro avg     0.9945    0.9945    0.9945     62002
weighted avg     0.9945    0.9945    0.9945     62002

released (GPU) correct: 61658 | windows that differ from the released predictions: 0
